# Task 1 Data Cleaning & Preprocessing

 

**Internship:** Auspify Technologies  
**Dataset:** Netflix Movies and TV Shows

## Objective
To clean, transform, and prepare the Netflix dataset
for further data analysis and machine learning.

In [2]:
# Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
# Load the Netflix dataset

df = pd.read_csv("Dataset.csv")

# Display first five records
df.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [4]:
# Check number of rows and columns

print("Dataset Shape:", df.shape)

# Check column names
print("\nColumn Names:")
print(df.columns.tolist())

# Check data types and non-null counts
print("\nDataset Information:")
df.info()

Dataset Shape: (8790, 10)

Column Names:
['show_id', 'type', 'title', 'director', 'country', 'date_added', 'release_year', 'rating', 'duration', 'listed_in']

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8790 non-null   object
 1   type          8790 non-null   object
 2   title         8790 non-null   object
 3   director      8790 non-null   object
 4   country       8790 non-null   object
 5   date_added    8790 non-null   object
 6   release_year  8790 non-null   int64 
 7   rating        8790 non-null   object
 8   duration      8790 non-null   object
 9   listed_in     8790 non-null   object
dtypes: int64(1), object(9)
memory usage: 686.8+ KB


In [5]:
# Check missing values

print("Missing Values:")
print(df.isnull().sum())

# Check total missing values
print("\nTotal Missing Values:", df.isnull().sum().sum())

# Check duplicate records
print("\nDuplicate Records:", df.duplicated().sum())

Missing Values:
show_id         0
type            0
title           0
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64

Total Missing Values: 0

Duplicate Records: 0


## 6. Data Cleaning and Handling Missing Values

In this step, we identify missing values, standardize text columns,
and handle inconsistent formatting in the Netflix dataset.

In [6]:
# Create a copy of the original dataset

df_clean = df.copy()

# Check missing values
print("Missing values before cleaning:")
print(df_clean.isnull().sum())

# Check empty strings
print("\nEmpty strings before cleaning:")

for col in df_clean.select_dtypes(include='object').columns:
    print(col, (df_clean[col].astype(str).str.strip() == '').sum())

Missing values before cleaning:
show_id         0
type            0
title           0
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64

Empty strings before cleaning:
show_id 0
type 0
title 0
director 0
country 0
date_added 0
rating 0
duration 0
listed_in 0


In [7]:
# Remove leading and trailing whitespace

text_columns = df_clean.select_dtypes(include='object').columns

for col in text_columns:
    df_clean[col] = df_clean[col].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

# Replace empty strings with NaN

df_clean.replace(r'^\s*$', np.nan, regex=True, inplace=True)

print("Text standardization completed.")

Text standardization completed.


In [8]:
# Fill missing values in selected categorical columns

df_clean['director'] = df_clean['director'].fillna('Unknown')
df_clean['country'] = df_clean['country'].fillna('Unknown')
df_clean['rating'] = df_clean['rating'].fillna('Not Rated')

# Check remaining missing values

print(df_clean.isnull().sum())

show_id         0
type            0
title           0
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64


In [9]:
# Convert date_added to datetime

df_clean['date_added'] = pd.to_datetime(
    df_clean['date_added'],
    format='%m/%d/%Y',
    errors='coerce'
)

# Verify the conversion

print("Date column datatype:", df_clean['date_added'].dtype)

print("\nDate samples:")
print(df_clean['date_added'].head())

Date column datatype: datetime64[ns]

Date samples:
0   2021-09-25
1   2021-09-24
2   2021-09-24
3   2021-09-22
4   2021-09-24
Name: date_added, dtype: datetime64[ns]


In [10]:
# Standardize placeholder values

placeholder_columns = ['director', 'country']

for col in placeholder_columns:
    df_clean[col] = df_clean[col].replace(
        'Not Given', 'Unknown'
    )

# Standardize content type formatting

df_clean['type'] = df_clean['type'].str.strip().str.title()

# Check categorical values

print("Content Types:")
print(df_clean['type'].value_counts())

print("\nRatings:")
print(df_clean['rating'].value_counts())

Content Types:
type
Movie      6126
Tv Show    2664
Name: count, dtype: int64

Ratings:
rating
TV-MA       3205
TV-14       2157
TV-PG        861
R            799
PG-13        490
TV-Y7        333
TV-Y         306
PG           287
TV-G         220
NR            79
G             41
TV-Y7-FV       6
NC-17          3
UR             3
Name: count, dtype: int64


In [11]:
# Extract numeric duration

df_clean['duration_value'] = pd.to_numeric(
    df_clean['duration'].str.extract(r'(\d+)')[0],
    errors='coerce'
)

# Extract duration unit

df_clean['duration_unit'] = df_clean['duration'].str.extract(
    r'([A-Za-z]+)'
)[0]

# Display results

df_clean[['type', 'duration', 'duration_value',
          'duration_unit']].head(10)

,type,duration,duration_value,duration_unit
0,Movie,90 min,90,min
1,Tv Show,1 Season,1,Season
2,Tv Show,1 Season,1,Season
3,Movie,91 min,91,min
4,Movie,125 min,125,min
5,Tv Show,9 Seasons,9,Seasons
6,Movie,104 min,104,min
7,Movie,87 min,87,min
8,Movie,127 min,127,min
9,Movie,76 min,76,min


## 8. Final Data Validation

In this step, we verify the cleaned dataset by checking:
- Missing values
- Duplicate records
- Data types
- Invalid dates
- Invalid duration values
- Dataset dimensions

In [12]:
# Check final dataset shape

print("Final Dataset Shape:", df_clean.shape)

# Check missing values

print("\nMissing Values:")
print(df_clean.isnull().sum())

# Check duplicates

print("\nDuplicate Records:", df_clean.duplicated().sum())

# Check data types

print("\nData Types:")
print(df_clean.dtypes)

# Check invalid dates

print("\nInvalid Dates:",
      df_clean['date_added'].isna().sum())

# Check invalid durations

print("Invalid Durations:",
      df_clean['duration_value'].isna().sum())

# Check content types

print("\nContent Type Distribution:")
print(df_clean['type'].value_counts())

Final Dataset Shape: (8790, 12)

Missing Values:
show_id           0
type              0
title             0
director          0
country           0
date_added        0
release_year      0
rating            0
duration          0
listed_in         0
duration_value    0
duration_unit     0
dtype: int64

Duplicate Records: 0

Data Types:
show_id                   object
type                      object
title                     object
director                  object
country                   object
date_added        datetime64[ns]
release_year               int64
rating                    object
duration                  object
listed_in                 object
duration_value             int64
duration_unit             object
dtype: object

Invalid Dates: 0
Invalid Durations: 0

Content Type Distribution:
type
Movie      6126
Tv Show    2664
Name: count, dtype: int64


In [13]:
# Create output folder

os.makedirs("output", exist_ok=True)

# Export cleaned dataset

df_clean.to_csv(
    "output/netflix_cleaned.csv",
    index=False
)

print("Cleaned dataset exported successfully!")

Cleaned dataset exported successfully!


In [14]:
# Reload the cleaned CSV

df_final = pd.read_csv("output/netflix_cleaned.csv")

print("Exported Dataset Shape:", df_final.shape)

df_final.head()

Exported Dataset Shape: (8790, 12)


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in,duration_value,duration_unit
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021-09-25,2020,PG-13,90 min,Documentaries,90,min
1,s3,Tv Show,Ganglands,Julien Leclercq,France,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",1,Season
2,s6,Tv Show,Midnight Mass,Mike Flanagan,United States,2021-09-24,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries",1,Season
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021-09-22,2021,TV-PG,91 min,"Children & Family Movies, Comedies",91,min
4,s8,Movie,Sankofa,Haile Gerima,United States,2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies",125,min


## Conclusion

The Netflix dataset was successfully cleaned and preprocessed.

Key activities performed:

1. Inspected the original dataset.
2. Identified missing values and duplicate records.
3. Standardized text and placeholder values.
4. Converted date_added into datetime format.
5. Transformed duration into numeric and categorical features.
6. Validated the cleaned dataset.
7. Exported the final dataset as netflix_cleaned.csv.

The cleaned dataset is now ready for Exploratory Data Analysis (EDA)
and further Data Science tasks.